#### Build Multi Agent With LangChain

In [52]:
# Import Packages
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage, ToolMessage
from tavily import TavilyClient
from dotenv.ipython import load_dotenv
from IPython.display import Markdown
import os
from pprint import pprint

In [53]:
# Import ApIKeys
load_dotenv(".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [54]:
# Prepare The MIND - LLMs
basic_mind = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, max_tokens=4696, api_key=GOOGLE_API_KEY)
advanced_mind = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_tokens=4696, api_key=GOOGLE_API_KEY)

In [55]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [56]:
# Prepare The Body - Tools

@tool
def getClientData(name: str):
    # Search on Data Base About Client
    '''
        Get Client Information About Marketing
    '''
    data = {
        "client_id": 12548,
        "fullName": name,
        "age": 34,
        "gender": "Male",
        "city": "Casablanca",
        "phone": "+212612345678",
        "email": "ahmed.elamrani@gmail.com",

        "customer_since": "2021-03-15",
        "last_purchase_date": "2026-06-20",
        "total_orders": 18,
        "total_spent": 24500,
        "average_order_value": 1361,

        "favorite_category": "Electronics",
        "favorite_brand": "Samsung",
        "purchase_frequency": "Monthly",

        "last_products": [
            "Samsung Galaxy S25",
            "Galaxy Buds Pro",
            "Wireless Charger"
        ],

        "loyalty_points": 3200,
        "membership": "Gold",

        "preferred_channel": "WhatsApp",
        "marketing_opt_in": True
    }
    return data


@tool
def getBestPracticesToGeterateMarketingOffre(query: str):
    '''
        Return List of Best Practices for generate marketing offre
    '''
    try:
        results = tavily_client.search(query=query)
        formatted_result = []
        for i, content in enumerate(results.get("results")):
            formatted_result.append(f"url: {content.get("url")}, title: {content.get("title")}, content: {content.get("content")}")
        return {"results": "\n".join(formatted_result)}
    except:
        return {"results": "WebSearch Tool Can Not get Informations"}


In [57]:
# Create Agent
agent = create_agent(
    model=advanced_mind,
    tools=[getClientData, getBestPracticesToGeterateMarketingOffre],\
    system_prompt="you are helpfull assistance. give response only using tools. if there no asnwer give 'Sorry I don't know!'",
    debug=True
)

In [65]:
# Ask Agent
config = {"configurable": {"thread_id": 1}}
res = agent.invoke(
    input={"messages": [HumanMessage("Generate Marketing Offre for Abdellah Karani")]},
    config=config
)
pprint(res["messages"][0].content)

[values] {'messages': [HumanMessage(content='Generate Marketing Offre for Abdellah Karani', additional_kwargs={}, response_metadata={}, id='19e3d7e5-d04d-466e-a33d-6403988e449f')]}
[updates] {'model': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Sorry I don't know!", 'extras': {'signature': 'CssGARFNMg92iQPb/AZZ5/AY2I/aHOut8G9+XGnJ0mQL766r7Z3Wizu2JEgtx2Sf+JnatCNb5GHhfzY79GfIpKx/4+1NTovs1RLVf2xxSTbEJOXzAXMZUSaGXolpN5L8vZlGd1n6eShd0dLoFrQvSvN2jubhNI7U+ugZot+y5mIuY+ZvnhiFI3/rtpVLmDSXUgnVzja05HBsgfCeTe9yRmDX1itQs9TURVqpYTAdV+5c8Y9IiFHUDHG5ijKlCiMyUu8X5K59lZE4H3yEfqmVUKzG9r3LbNBfCMnxfJ9fkANlsxeZi/A4SPp+g4cf+ZIV5TAQwHGmwmOuz97wzhxcDKL4a/b/kFfAfZYVtlGFxomP2XP5WfNElt46y0LxsosQNbIVq3tu3bCeu4vwf00u6jOe6R1MXa1SnXdaXTjUKL1Li6b/Fgyj83RJULY9L92dWB3KgACH/aBjofztap18huk+8eC0ZqiTEuSs+voE14jn8GEKGN5fYpBDD/v0GsuWRkQFkBNXDpGRkyikYl4ss8EnDjHDfi6WsRQ9mRaI/TTqESQPADSTH5HOimYYqkIk9iYqrt2m9nMGWMR9XYK/F5liY0b7jA635/qcVJEkL99q0VnnaJIl2Cs2zvZb7fFybltvFqd98TIo3opUKRlipc55CyOyO8xID2ZVCStM6U+wl0BjZdHbh